In [1]:
# ============================================================
# NOTEBOOK — Level 1 Fine-Tuning Robustness Check (v3, SINGLE T4)
# Dataset: nazmusresan/fitzpatrick17k
# GPU T4 x1, Internet ON
# Expected runtime: ~3-4 hours per seed × 5 seeds = 15-20 hours total
#
# CHANGES FROM DUAL-GPU VERSION
# -----------------------------
# 1. Pin to GPU 0 only via CUDA_VISIBLE_DEVICES (set BEFORE importing torch)
# 2. DataParallel disabled (USE_DP forced False)
# 3. BATCH_SIZE halved 16→8 to avoid OOM on a single 16GB T4
# 4. GRAD_ACCUM doubled 2→4 so the EFFECTIVE batch size stays at 32
#    (8 per-step × 4 accum = 32, identical to the dual-GPU run's
#     16 per-GPU × 2 GPUs × 1 accum = 32, give or take BN/dropout noise)
# 5. num_workers reduced 4→2 to lower CPU/RAM pressure
# 6. eval batch size halved for memory safety during predict()
# 7. Aggressive cleanup (gc.collect + empty_cache) between runs
#
# Everything else — the bug fix, the seeds, eta=0.01, the pool size of 200,
# the plain shuffled DRO batching, the SMOTE re-normalization — is identical
# to v3 dual-GPU. The intent is that running this gives the same results
# as the dual-GPU notebook would have, just slower.
#
# If you've already completed seeds 42 and 0 in the dual-GPU run, you can
# skip them here by editing CFG['SEEDS'] to only contain the missing ones.
# ============================================================

# CRITICAL: pin to GPU 0 BEFORE importing torch
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

!pip install transformers torch torchvision scikit-learn pandas numpy imbalanced-learn Pillow -q

import gc
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from imblearn.over_sampling import SMOTE
from sklearn.metrics import confusion_matrix, roc_auc_score
from torch.utils.data import ConcatDataset, DataLoader, Dataset, Subset
from transformers import CLIPModel, CLIPProcessor

warnings.filterwarnings('ignore')

# ── SINGLE GPU SETUP ──────────────────────────────────────────
n_gpus = torch.cuda.device_count()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPUs visible: {n_gpus} (CUDA_VISIBLE_DEVICES=0)")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Total memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
USE_DP = False
print(f"DataParallel: DISABLED (single-GPU run)")

# ── CONFIG ────────────────────────────────────────────────────
CFG = dict(
    FITZ_CSV          = '/kaggle/input/datasets/nazmusresan/fitzpatrick17k/New folder/fitzpatrick17k (1).csv',
    IMG_DIR           = '/kaggle/input/datasets/nazmusresan/fitzpatrick17k/New folder/background removed',
    RESULTS_DIR       = 'results_level1',
    RAW_PREDS_DIR     = 'results_level1/raw_preds',
    CLASS_LABELS      = ['non-neoplastic', 'benign', 'malignant'],

    # Match nb_mega exactly
    SEEDS             = [42, 0, 1, 7, 99],
    GDRO_ETA          = 0.01,
    REAL_OVERSAMPLE_N = 200,

    # Fine-tuning hyperparameters
    FT_LAST_N_BLOCKS  = 4,
    BATCH_SIZE        = 8,            # halved from 16 for single-GPU memory
    GRAD_ACCUM        = 4,            # doubled from 2 to keep effective=32
    EVAL_BATCH_SIZE   = 16,           # smaller eval batch for safety
    EPOCHS            = 5,
    LR_HEAD           = 1e-4,
    LR_BACKBONE       = 1e-5,
    WEIGHT_DECAY      = 0.01,
    DROPOUT           = 0.3,
    MIXED_PRECISION   = True,
    SMOTE_HEAD_EPOCHS = 20,
    N_BOOTSTRAP       = 1000,
)

for d in [CFG['RESULTS_DIR'], CFG['RAW_PREDS_DIR']]:
    os.makedirs(d, exist_ok=True)

print(f"\nEffective batch size: {CFG['BATCH_SIZE']} × {CFG['GRAD_ACCUM']} = "
      f"{CFG['BATCH_SIZE'] * CFG['GRAD_ACCUM']}")
print(f"(Matches dual-GPU run's effective batch of 16 × 2 GPUs × 1 = 32)")

# ── AMP ───────────────────────────────────────────────────────
def make_autocast():
    return torch.amp.autocast('cuda') if torch.cuda.is_available() else torch.amp.autocast('cpu')

def make_scaler():
    return torch.amp.GradScaler('cuda') if CFG['MIXED_PRECISION'] and torch.cuda.is_available() else None

# ── Load data ─────────────────────────────────────────────────
print('\nLoading Fitzpatrick17k metadata...')
df = pd.read_csv(CFG['FITZ_CSV'])
df = df[df['fitzpatrick_scale'].notna() & (df['fitzpatrick_scale'] > 0)]
df = df[df['three_partition_label'].isin(CFG['CLASS_LABELS'])]

image_files = {
    f.replace('.jpg', '').replace('.png', ''): os.path.join(CFG['IMG_DIR'], f)
    for f in os.listdir(CFG['IMG_DIR'])
    if f.endswith('.jpg') or f.endswith('.png')
}
df['local_path'] = df['md5hash'].map(image_files)
df = df[df['local_path'].notna()].copy()

class_map = {name: i for i, name in enumerate(CFG['CLASS_LABELS'])}
df['target'] = df['three_partition_label'].map(class_map)
df['fitzpatrick_scale'] = df['fitzpatrick_scale'].astype(int)

light_df = df[df['fitzpatrick_scale'].isin([1, 2])].copy()
dark_df  = df[df['fitzpatrick_scale'].isin([5, 6])].copy()
print(f'Total valid images: {len(df)}')
print(f'Light (Fitz I-II) train pool: {len(light_df)}')
print(f'Dark (Fitz V-VI) full set:    {len(dark_df)}')
print(f'  └─ benign: {(dark_df["target"]==1).sum()} '
      f'({(dark_df["target"]==1).sum()/len(dark_df)*100:.1f}%)')

assert len(dark_df) == 2168, f'Expected 2168 dark images, got {len(dark_df)}'
assert (dark_df['target'] == 1).sum() == 203, \
    f'Expected 203 dark benign, got {(dark_df["target"]==1).sum()}'
print('✓ Splits match paper (n=2168 dark, 203 benign)')

# ── Dataset ───────────────────────────────────────────────────
class SkinDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.processor = processor
        self.images, self.labels, self.fitz = [], [], []
        failed = 0
        for _, row in dataframe.reset_index(drop=True).iterrows():
            try:
                img = Image.open(row['local_path']).convert('RGB')
                self.images.append(img)
                self.labels.append(int(row['target']))
                self.fitz.append(int(row['fitzpatrick_scale']))
            except Exception:
                failed += 1
        print(f'  Loaded {len(self.images)} images ({failed} failed)')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        inputs = self.processor(images=self.images[idx], return_tensors='pt')
        return inputs['pixel_values'].squeeze(0), torch.tensor(self.labels[idx], dtype=torch.long)


class GroupedDataset(Dataset):
    def __init__(self, base_ds, group_labels):
        self.base = base_ds
        self.groups = np.asarray(group_labels)
        assert len(base_ds) == len(group_labels)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        px, lbl = self.base[idx]
        return px, lbl, torch.tensor(self.groups[idx], dtype=torch.long)

# ── Model ─────────────────────────────────────────────────────
class CLIPFineTuned(nn.Module):
    def __init__(self, clip_model, num_classes=3, dropout=0.3, ft_last_n=4):
        super().__init__()
        self.vision_model      = clip_model.vision_model
        self.visual_projection = clip_model.visual_projection
        hidden_size = clip_model.config.projection_dim

        for p in self.vision_model.parameters():
            p.requires_grad = False
        for p in self.visual_projection.parameters():
            p.requires_grad = True

        n_layers = len(self.vision_model.encoder.layers)
        for i in range(n_layers - ft_last_n, n_layers):
            for p in self.vision_model.encoder.layers[i].parameters():
                p.requires_grad = True
        for p in self.vision_model.post_layernorm.parameters():
            p.requires_grad = True

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, pixel_values):
        out   = self.vision_model(pixel_values=pixel_values)
        feats = self.visual_projection(out.pooler_output)
        feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-8)
        return self.classifier(feats)

    def get_features(self, pixel_values):
        out   = self.vision_model(pixel_values=pixel_values)
        feats = self.visual_projection(out.pooler_output)
        feats = feats / (feats.norm(dim=-1, keepdim=True) + 1e-8)
        return feats

def wrap_model(model):
    return model.to(device)  # no DataParallel on single GPU

def unwrap(model):
    return model  # nothing to unwrap

def make_param_groups(model):
    base = unwrap(model)
    backbone_params = [p for n, p in base.named_parameters()
                       if p.requires_grad and ('vision_model' in n or 'visual_projection' in n)]
    head_params     = [p for n, p in base.named_parameters()
                       if p.requires_grad and 'classifier' in n]
    return [
        {'params': backbone_params, 'lr': CFG['LR_BACKBONE']},
        {'params': head_params,     'lr': CFG['LR_HEAD']},
    ]

# ── Helpers ───────────────────────────────────────────────────
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (float('nan'), float('nan'))
    p      = k / n
    denom  = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * (p * (1 - p) / n + z**2 / (4 * n**2)) ** 0.5 / denom
    return max(0.0, center - margin), min(1.0, center + margin)

def bootstrap_auc_ci(y_true, y_score, n_boot=1000, seed=42):
    rng  = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        idx = rng.choice(len(y_true), len(y_true), replace=True)
        try:
            aucs.append(roc_auc_score(y_true[idx], y_score[idx], multi_class='ovr'))
        except ValueError:
            continue
    return tuple(np.percentile(aucs, [2.5, 97.5])) if aucs else (float('nan'), float('nan'))

def evaluate_full(y_true, y_proba, y_pred):
    classes_present = np.unique(y_true)
    if len(classes_present) < y_proba.shape[1]:
        per_class_aucs = []
        for c in classes_present:
            y_binary = (y_true == c).astype(int)
            try:
                per_class_aucs.append(roc_auc_score(y_binary, y_proba[:, c]))
            except ValueError:
                pass
        auc = float(np.mean(per_class_aucs)) if per_class_aucs else float('nan')
    else:
        auc = roc_auc_score(y_true, y_proba, multi_class='ovr')
    res = {'auc': auc}
    for c, name in [(0, 'non_neo'), (1, 'benign'), (2, 'malignant')]:
        m = (y_true == c)
        res[f'acc_{name}_dark'] = float((y_pred[m] == c).mean()) if m.sum() > 0 else float('nan')
    return res

@torch.no_grad()
def predict(model, dataloader):
    model.eval()
    all_probs, all_labels = [], []
    for pixel_values, labels in dataloader:
        pixel_values = pixel_values.to(device)
        if CFG['MIXED_PRECISION']:
            with make_autocast():
                logits = model(pixel_values)
        else:
            logits = model(pixel_values)
        probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
    return np.vstack(all_probs), np.concatenate(all_labels)

@torch.no_grad()
def extract_features(model, dataloader):
    model.eval()
    base = unwrap(model)
    all_feats, all_labels = [], []
    for pixel_values, labels in dataloader:
        pixel_values = pixel_values.to(device)
        if CFG['MIXED_PRECISION']:
            with make_autocast():
                feats = base.get_features(pixel_values)
        else:
            feats = base.get_features(pixel_values)
        all_feats.append(feats.float().cpu().numpy())
        all_labels.append(labels.numpy())
    return np.vstack(all_feats), np.concatenate(all_labels)

def _flush(scaler, optimizer):
    if scaler is not None:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()
    optimizer.zero_grad()

# ── Training: baseline ────────────────────────────────────────
def train_baseline(model, dataloader, n_epochs):
    optimizer = optim.AdamW(make_param_groups(model), weight_decay=CFG['WEIGHT_DECAY'])
    criterion = nn.CrossEntropyLoss()
    scaler    = make_scaler()

    for epoch in range(n_epochs):
        model.train()
        total_loss, n_batches, pending = 0.0, 0, 0
        optimizer.zero_grad()
        for pixel_values, labels in dataloader:
            pixel_values, labels = pixel_values.to(device), labels.to(device)
            if CFG['MIXED_PRECISION']:
                with make_autocast():
                    loss = criterion(model(pixel_values), labels) / CFG['GRAD_ACCUM']
                scaler.scale(loss).backward()
            else:
                loss = criterion(model(pixel_values), labels) / CFG['GRAD_ACCUM']
                loss.backward()
            pending += 1
            if pending == CFG['GRAD_ACCUM']:
                _flush(scaler, optimizer)
                pending = 0
            total_loss += loss.item() * CFG['GRAD_ACCUM']
            n_batches  += 1
        if pending > 0:
            _flush(scaler, optimizer)
        print(f'    Epoch {epoch+1}/{n_epochs}  loss={total_loss/n_batches:.4f}')

# ── Training: Group DRO ───────────────────────────────────────
def train_group_dro(model, dataset, group_labels, n_epochs, n_groups,
                    batch_size, seed):
    print(f'    GDRO group counts: '
          f'{np.bincount(group_labels, minlength=n_groups).tolist()}')

    dataloader    = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                               num_workers=2, pin_memory=True)
    optimizer     = optim.AdamW(make_param_groups(model), weight_decay=CFG['WEIGHT_DECAY'])
    criterion     = nn.CrossEntropyLoss(reduction='none')
    scaler        = make_scaler()
    group_weights = torch.ones(n_groups, device=device) / n_groups

    for epoch in range(n_epochs):
        model.train()
        total_loss, n_batches, pending = 0.0, 0, 0
        optimizer.zero_grad()
        for pixel_values, labels, groups in dataloader:
            pixel_values = pixel_values.to(device)
            labels       = labels.to(device)
            groups       = groups.to(device)

            if CFG['MIXED_PRECISION']:
                with make_autocast():
                    logits       = model(pixel_values)
                    losses       = criterion(logits, labels)
                    group_losses = torch.zeros(n_groups, device=device)
                    for g in range(n_groups):
                        m = (groups == g)
                        if m.sum() > 0:
                            group_losses[g] = losses[m].mean()
                gl_fp32 = group_losses.detach().float()
                group_weights = group_weights * torch.exp(CFG['GDRO_ETA'] * gl_fp32)
                group_weights = group_weights / group_weights.sum()
                if not torch.isfinite(group_weights).all():
                    group_weights = torch.ones(n_groups, device=device) / n_groups
                weighted_loss = (group_weights * group_losses.float()).sum() / CFG['GRAD_ACCUM']
                scaler.scale(weighted_loss).backward()
            else:
                logits       = model(pixel_values)
                losses       = criterion(logits, labels)
                group_losses = torch.zeros(n_groups, device=device)
                for g in range(n_groups):
                    m = (groups == g)
                    if m.sum() > 0:
                        group_losses[g] = losses[m].mean()
                group_weights = group_weights * torch.exp(CFG['GDRO_ETA'] * group_losses.detach())
                group_weights = group_weights / group_weights.sum()
                if not torch.isfinite(group_weights).all():
                    group_weights = torch.ones(n_groups, device=device) / n_groups
                weighted_loss = (group_weights * group_losses).sum() / CFG['GRAD_ACCUM']
                weighted_loss.backward()

            pending += 1
            if pending == CFG['GRAD_ACCUM']:
                _flush(scaler, optimizer)
                pending = 0
            total_loss += weighted_loss.item() * CFG['GRAD_ACCUM']
            n_batches  += 1

        if pending > 0:
            _flush(scaler, optimizer)
        print(f'    Epoch {epoch+1}/{n_epochs}  loss={total_loss/n_batches:.4f}  '
              f'group_w=[{",".join(f"{w:.3f}" for w in group_weights.cpu().numpy())}]')

# ── Build datasets ────────────────────────────────────────────
print('\nLoading CLIP processor...')
processor = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')

print('\nBuilding datasets (loading images into RAM)...')
print('  Light (train pool):')
light_ds = SkinDataset(light_df, processor)
print('  Dark (full set, split per seed):')
dark_ds  = SkinDataset(dark_df, processor)

# ── Pool / test split ─────────────────────────────────────────
def make_pool_split(seed, n_pool=CFG['REAL_OVERSAMPLE_N']):
    rng_local   = np.random.default_rng(seed)
    dark_labels = np.array(dark_ds.labels)
    pool_idx    = []
    for cls in [0, 1, 2]:
        cls_idx = np.where(dark_labels == cls)[0]
        n_take  = min(int(round(n_pool * len(cls_idx) / len(dark_labels))), len(cls_idx))
        pool_idx.extend(rng_local.choice(cls_idx, n_take, replace=False).tolist())
    pool_idx = np.array(sorted(pool_idx))
    test_idx = np.array([i for i in range(len(dark_ds)) if i not in set(pool_idx)])
    return pool_idx, test_idx

pool42, test42 = make_pool_split(42)
n_test_benign_ex = sum(1 for i in test42 if dark_ds.labels[i] == 1)
print(f'\nPool/test split (seed 42 example):')
print(f'  Pool: {len(pool42)} (drawn stratified from dark)')
print(f'  Test (interventions): {len(test42)} | benign in test: {n_test_benign_ex}')
print(f'  Test (baseline): 2168 (full dark set)')

# ══════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════
INTERVENTIONS = ['baseline', 'group_dro', 'smote']
all_results   = []
t_start       = time.time()

# Single-GPU: do NOT scale by n_gpus
dl_batch      = CFG['BATCH_SIZE']
eval_dl_batch = CFG['EVAL_BATCH_SIZE']

full_dark_loader = DataLoader(dark_ds, batch_size=eval_dl_batch, shuffle=False,
                              num_workers=2, pin_memory=True)

for seed in CFG['SEEDS']:
    print(f"\n{'='*60}\nSEED {seed}\n{'='*60}")
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    rng = np.random.default_rng(seed)

    pool_idx, test_idx = make_pool_split(seed)
    test_subset = Subset(dark_ds, test_idx.tolist())
    pool_subset = Subset(dark_ds, pool_idx.tolist())
    intv_test_loader = DataLoader(test_subset, batch_size=eval_dl_batch,
                                  shuffle=False, num_workers=2, pin_memory=True)

    n_test_benign_seed = sum(1 for i in test_idx if dark_ds.labels[i] == 1)
    print(f'  Pool: {len(pool_idx)} | Intervention test: {len(test_idx)} '
          f'(benign: {n_test_benign_seed}) | Baseline test: 2168')

    for intervention in INTERVENTIONS:
        print(f'\n── {intervention.upper()} (seed {seed}) ──')
        t0 = time.time()

        clip_base = CLIPModel.from_pretrained('openai/clip-vit-large-patch14')
        model     = CLIPFineTuned(clip_base, num_classes=3,
                                  dropout=CFG['DROPOUT'],
                                  ft_last_n=CFG['FT_LAST_N_BLOCKS'])
        model     = wrap_model(model)
        n_train   = sum(p.numel() for p in unwrap(model).parameters() if p.requires_grad)
        print(f'  Trainable params: {n_train/1e6:.1f}M  |  GPU: 1 (single)')
        if torch.cuda.is_available():
            print(f'  GPU mem after model load: '
                  f'{torch.cuda.memory_allocated()/1e9:.2f} GB')

        try:
            if intervention == 'baseline':
                train_dl = DataLoader(light_ds, batch_size=dl_batch,
                                      shuffle=True, num_workers=2, pin_memory=True)
                train_baseline(model, train_dl, CFG['EPOCHS'])
                eval_loader = full_dark_loader

            elif intervention == 'group_dro':
                combined     = ConcatDataset([light_ds, pool_subset])
                group_labels = np.array([0] * len(light_ds) + [1] * len(pool_subset))
                grouped      = GroupedDataset(combined, group_labels)
                train_group_dro(model, grouped, group_labels,
                                n_epochs=CFG['EPOCHS'], n_groups=2,
                                batch_size=dl_batch, seed=seed)
                eval_loader = intv_test_loader

            elif intervention == 'smote':
                train_dl = DataLoader(light_ds, batch_size=dl_batch,
                                      shuffle=True, num_workers=2, pin_memory=True)
                train_baseline(model, train_dl, CFG['EPOCHS'])

                print('    Extracting features for SMOTE...')
                light_feats, light_lbls = extract_features(
                    model, DataLoader(light_ds, batch_size=eval_dl_batch,
                                      shuffle=False, num_workers=2, pin_memory=True))
                dark_feats, dark_lbls = extract_features(
                    model, DataLoader(pool_subset, batch_size=eval_dl_batch,
                                      shuffle=False, num_workers=2, pin_memory=True))

                combo_feats = np.vstack([light_feats, dark_feats])
                combo_lbls  = np.concatenate([light_lbls, dark_lbls])
                print(f'    Pre-SMOTE:  {np.bincount(combo_lbls)}')
                try:
                    k = max(1, min(5, int(np.bincount(combo_lbls).min()) - 1))
                    smote_feats, smote_lbls = SMOTE(
                        random_state=seed, k_neighbors=k).fit_resample(combo_feats, combo_lbls)
                    norms = np.linalg.norm(smote_feats, axis=1, keepdims=True)
                    smote_feats = smote_feats / np.maximum(norms, 1e-8)
                    print(f'    Post-SMOTE: {np.bincount(smote_lbls)} (re-normalized)')
                except Exception as e:
                    print(f'    SMOTE failed ({e}) — using raw features')
                    smote_feats, smote_lbls = combo_feats, combo_lbls

                print('    Retraining classifier head on SMOTE features...')
                base = unwrap(model)
                for p in base.vision_model.parameters():
                    p.requires_grad = False
                for p in base.visual_projection.parameters():
                    p.requires_grad = False

                X        = torch.tensor(smote_feats, dtype=torch.float32).to(device)
                Y        = torch.tensor(smote_lbls,  dtype=torch.long).to(device)
                head_opt = optim.AdamW(base.classifier.parameters(),
                                       lr=CFG['LR_HEAD'], weight_decay=CFG['WEIGHT_DECAY'])
                criterion = nn.CrossEntropyLoss()
                base.classifier.train()

                for ep in range(CFG['SMOTE_HEAD_EPOCHS']):
                    perm    = torch.randperm(len(X))
                    ep_loss = 0.0
                    for i in range(0, len(X), 64):
                        idx = perm[i:i+64]
                        head_opt.zero_grad()
                        loss = criterion(base.classifier(X[idx]), Y[idx])
                        loss.backward()
                        head_opt.step()
                        ep_loss += loss.item()
                    if (ep + 1) % 5 == 0:
                        print(f'      Head epoch {ep+1}/{CFG["SMOTE_HEAD_EPOCHS"]}  loss={ep_loss:.3f}')

                # Free SMOTE feature tensors before eval
                del X, Y, smote_feats, smote_lbls, combo_feats, combo_lbls
                del light_feats, light_lbls, dark_feats, dark_lbls
                gc.collect()
                torch.cuda.empty_cache()

                eval_loader = intv_test_loader

            # ── Evaluate ──────────────────────────────────────────
            print('  Evaluating on dark test set...')
            proba, labels = predict(model, eval_loader)
            preds  = proba.argmax(axis=1)
            res    = evaluate_full(labels, proba, preds)
            n_b    = int((labels == 1).sum())
            k_b    = int(((preds == 1) & (labels == 1)).sum())
            w_lo, w_hi = wilson_ci(k_b, n_b)
            b_lo, b_hi = bootstrap_auc_ci(labels, proba, CFG['N_BOOTSTRAP'], seed)
            cm = confusion_matrix(labels, preds, labels=[0, 1, 2])

            print(f'  demo_auc={res["auc"]:.4f} (95% CI {b_lo:.4f}-{b_hi:.4f})')
            print(f'  acc_non_neo_dark   = {res["acc_non_neo_dark"]:.4f}')
            print(f'  acc_benign_dark    = {res["acc_benign_dark"]:.4f}  '
                  f'({k_b}/{n_b}, Wilson CI {w_lo:.4f}-{w_hi:.4f})')
            print(f'  acc_malignant_dark = {res["acc_malignant_dark"]:.4f}')
            print(f'  Confusion matrix (rows=true, cols=pred):')
            print(f'    non-neo: {cm[0]}')
            print(f'    benign:  {cm[1]}')
            print(f'    malig:   {cm[2]}')

            raw_dir = os.path.join(CFG['RAW_PREDS_DIR'], intervention, f'seed{seed}')
            os.makedirs(raw_dir, exist_ok=True)
            np.save(f'{raw_dir}/demo_y_true.npy',      labels)
            np.save(f'{raw_dir}/demo_y_pred.npy',       preds)
            np.save(f'{raw_dir}/demo_y_pred_proba.npy', proba)

            all_results.append({
                'seed':               seed,
                'intervention':       intervention,
                'demo_auc':           res['auc'],
                'demo_ci_lo':         b_lo,
                'demo_ci_hi':         b_hi,
                'acc_non_neo_dark':   res['acc_non_neo_dark'],
                'acc_benign_dark':    res['acc_benign_dark'],
                'acc_malignant_dark': res['acc_malignant_dark'],
                'benign_wilson_lo':   w_lo,
                'benign_wilson_hi':   w_hi,
                'n_dark_benign':      n_b,
                'n_dark_total':       len(labels),
            })

            # Incremental save: write CSV after EVERY run, so a crash
            # doesn't lose previously-completed seeds.
            pd.DataFrame(all_results).to_csv(
                os.path.join(CFG['RESULTS_DIR'], 'level1_finetune_results.csv'),
                index=False)

            print(f'  ✓ Done in {(time.time()-t0)/60:.1f} min  '
                  f'(total: {(time.time()-t_start)/60:.1f} min)')

        except Exception as e:
            print(f'  ✗ Run FAILED: {e}')
            import traceback; traceback.print_exc()

        finally:
            # Aggressive cleanup between runs
            del model, clip_base
            gc.collect()
            torch.cuda.empty_cache()
            if torch.cuda.is_available():
                print(f'  GPU mem after cleanup: '
                      f'{torch.cuda.memory_allocated()/1e9:.2f} GB')

# ══════════════════════════════════════════════════════════════
# SAVE AND SUMMARIZE
# ══════════════════════════════════════════════════════════════
print(f"\n{'='*60}\nSUMMARY\n{'='*60}")
df_results = pd.DataFrame(all_results)

n_expected = len(CFG['SEEDS']) * len(INTERVENTIONS)
if len(df_results) < n_expected:
    missing = n_expected - len(df_results)
    print(f'WARNING: {missing}/{n_expected} runs failed — summary is partial.')
    completed = {(r['seed'], r['intervention']) for r in all_results}
    for s in CFG['SEEDS']:
        for iv in INTERVENTIONS:
            if (s, iv) not in completed:
                print(f'  MISSING: seed={s}, intervention={iv}')
else:
    print(f'✓ All {n_expected} runs completed')

df_results.to_csv(os.path.join(CFG['RESULTS_DIR'], 'level1_finetune_results.csv'), index=False)
print(f'Saved: {CFG["RESULTS_DIR"]}/level1_finetune_results.csv')

print('\n── Mean ± std across seeds ──')
summary = df_results.groupby('intervention').agg(
    demo_auc_mean      = ('demo_auc',          'mean'),
    demo_auc_std       = ('demo_auc',          'std'),
    benign_acc_mean    = ('acc_benign_dark',    'mean'),
    benign_acc_std     = ('acc_benign_dark',    'std'),
    malignant_acc_mean = ('acc_malignant_dark', 'mean'),
    malignant_acc_std  = ('acc_malignant_dark', 'std'),
    non_neo_acc_mean   = ('acc_non_neo_dark',   'mean'),
    non_neo_acc_std    = ('acc_non_neo_dark',   'std'),
).round(4)
print(summary.to_string())

total = time.time() - t_start
print(f'\n✓ ALL DONE in {total/60:.1f} min ({total/3600:.2f} h)')
print('\nPaste this entire output back to Claude.')

Device: cuda
GPUs visible: 1 (CUDA_VISIBLE_DEVICES=0)
  GPU: Tesla T4
  Total memory: 15.6 GB
DataParallel: DISABLED (single-GPU run)

Effective batch size: 8 × 4 = 32
(Matches dual-GPU run's effective batch of 16 × 2 GPUs × 1 = 32)

Loading Fitzpatrick17k metadata...
Total valid images: 16012
Light (Fitz I-II) train pool: 7755
Dark (Fitz V-VI) full set:    2168
  └─ benign: 203 (9.4%)
✓ Splits match paper (n=2168 dark, 203 benign)

Loading CLIP processor...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/905 [00:00<?, ?B/s]

...

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Trainable params: 51.4M  |  GPU: 1 (single)
  GPU mem after model load: 2.46 GB
    GDRO group counts: [7755, 200]
    Epoch 1/5  loss=0.7178  group_w=[0.999,0.001]
    Epoch 2/5  loss=0.6304  group_w=[1.000,0.000]
    Epoch 3/5  loss=0.5184  group_w=[1.000,0.000]
    Epoch 4/5  loss=0.3758  group_w=[1.000,0.000]
    Epoch 5/5  loss=0.2672  group_w=[1.000,0.000]
  Evaluating on dark test set...
  demo_auc=0.7281 (95% CI 0.6994-0.7592)
  acc_non_neo_dark   = 0.8169
  acc_benign_dark    = 0.3315  (61/184, Wilson CI 0.2676-0.4024)
  acc_malignant_dark = 0.4656
  Confusion matrix (rows=true, cols=pred):
    non-neo: [1303  232   60]
    benign:  [102  61  21]
    malig:   [81 20 88]
  ✓ Done in 12.5 min  (total: 178.4 min)
  GPU mem after cleanup: 1.24 GB

── SMOTE (seed 99) ──


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Trainable params: 51.4M  |  GPU: 1 (single)
  GPU mem after model load: 2.46 GB
    Epoch 1/5  loss=0.7911
    Epoch 2/5  loss=0.6229
    Epoch 3/5  loss=0.4962
    Epoch 4/5  loss=0.3267
    Epoch 5/5  loss=0.2174
    Extracting features for SMOTE...
    Pre-SMOTE:  [5607 1134 1214]
    Post-SMOTE: [5607 5607 5607] (re-normalized)
    Retraining classifier head on SMOTE features...
      Head epoch 5/20  loss=58.202
      Head epoch 10/20  loss=54.572
      Head epoch 15/20  loss=51.843
      Head epoch 20/20  loss=49.070
  Evaluating on dark test set...
  demo_auc=0.7013 (95% CI 0.6696-0.7292)
  acc_non_neo_dark   = 0.8420
  acc_benign_dark    = 0.3207  (59/184, Wilson CI 0.2575-0.3912)
  acc_malignant_dark = 0.3598
  Confusion matrix (rows=true, cols=pred):
    non-neo: [1343  203   49]
    benign:  [111  59  14]
    malig:   [87 34 68]
  ✓ Done in 13.9 min  (total: 192.3 min)
  GPU mem after cleanup: 1.24 GB

SUMMARY
✓ All 15 runs completed
Saved: results_level1/level1_finetune_r